# Per-pore analysis from XCT volumes (Defect Atlas backend)

Computes, for **every pore** in a specimen, geometric / shape / orientation / distance-to-surface / border / classification metrics, plus specimen-level summaries (void volume fraction, counts by type, key figures, histograms).

**Nothing is reimplemented.** The notebook imports the existing Defect Atlas backend (`app.…`, read-only) and calls:

| Step | Backend function |
|---|---|
| labelling + all properties + border flags | `PipelineExecutor.run_pipeline` |
| Pore / LoF / Cluster / Border | `app.domain.services.classification.classify_defects` |
| shape class (Spherical / Disc / …) | `app.domain.services.shape_classifier.classify_shape` (inputs converted with `app.routers.study._covariance_ratio_to_axis_ratio`) |
| key figures | `app.services.analytics_service._volume_key_figures` |
| per-slice porosity | `app.services.porosity_report_service.compute_direction_report` |
| Z projections | `app.domain.services.projections.compute_projections` |

**Inputs are TIFF** (`*_onlypores.tif`, `*_samplemask.tif`, as written by `onlypores.ipynb`). The backend only reads RAW, so they are streamed slice-by-slice into polarity-normalised RAW files in `OUT_DIR/_raw/` (pores = 255, material = 0). Axis convention is the TIFF's: `(Z, Y, X)` = `(page, row, column)`.

**Kernel:** `defect-atlas` (Python 3.11, pydantic 1.x, numpy 1.26). The default `prep` kernel cannot import the backend.

## 1. Parameters

In [6]:
# ----------------------------- INPUTS ---------------------------------------
PORES_VOLUME_PATH = "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_01_1/onlypores files/volume_eq_aligned_onlypores.tif"
MASK_VOLUME_PATH  = "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_01_1/onlypores files/volume_eq_aligned_samplemask.tif"
GRAY_VOLUME_PATH  = None       # optional; only shape-checked and recorded, no backend property uses it

VOXEL_SIZE_MM = 0.025          # isotropic voxel size in mm (0.025 mm = 25 µm)  <-- SET PER SPECIMEN
INVERT_PORES  = None           # None = auto-detect polarity; True/False = force
INVERT_MASK   = None           # (polarity = which value means "pore"/"material")
WALLS = "auto"                 # front/back wall exclusion along the TIFF Z axis (pages):
                               #   "auto" = read 'Front wall slice' / 'Back wall slice' from the *_report.txt next to the pores TIFF
                               #   (front, back) = explicit slice indices (keeps front <= z < back);  None = no wall exclusion
CROP = None                    # None, or ((z0, z1), (y0, y1), (x0, x1)) in voxels, for quick tests

# ----------------------------- PIPELINE -------------------------------------
MIN_SIZE_VOXELS = 0            # kill_small   (0 = keep everything)
MAX_SIZE_VOXELS = None         # kill_big     (None = no limit)
CONNECTIVITY    = 26
ENABLE_MERGING  = False        # merge nearby defects into clusters (recomputes properties)
ENABLE_LOF      = True         # False => no pore can become LoF (only Pore/Cluster/Border)

# classification (classify_defects defaults)
LOF_VOLUME_THRESHOLD_VOX   = 60.0
LOF_ELONGATION_THRESHOLD   = 0.7
LOF_DISTANCE_THRESHOLD_VOX = 100.0
CPX_THRESHOLD = 0.7001         # classify_shape default: cpx_factor above this => "Complex"

# ----------------------------- OUTPUT / RUN ---------------------------------
OUT_DIR = None                 # None => <pores folder>/pore_analysis
USE_CACHE = True               # reuse raw conversion + pipeline result if inputs/params unchanged
KEEP_RAW = True                # keep OUT_DIR/_raw/*.raw (needed for cache reuse, ~1 byte/voxel each)
RUN_SLICE_PROFILE = True
SLICE_AXES = ("z", "y", "x")   # per-slice porosity directions
RUN_PROJECTION = True

# ----------------------------- SETUP PATHS ----------------------------------
REPO = "/home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/Defect Atlas"

## 2. Environment and import setup

* Sets `DEFECT_ATLAS_ACCELERATION_MODE=cpu` *before* importing (no `cupy` in this env, so the CUDA path would only fail and fall back).
* Sets `sys.dont_write_bytecode` so **no `__pycache__` is written inside `REPO`** (read-only use).
* Checks that the whole `REPO` tree is intact: `app/core/supernames.py` inserts `v1-tk-version/DefViz-main` into `sys.path` and imports `quantification_analyzer`.
* No `.env` is needed: `app.core.config` has defaults for every setting.
* Importing any `app.domain.services.*` eagerly imports all services – slow, expected. The "Legacy … volumelab not available" messages are harmless (legacy DefQ package is optional).
* **Memory:** labelling + EDT need roughly 8–12 bytes per voxel (int32 labels, float32 distance map; the backend switches to slab labelling / out-of-core EDT automatically for big volumes). Properties are then computed per pore in threads.

In [7]:
import os, sys, json, hashlib, time, math, tempfile, subprocess, warnings
sys.dont_write_bytecode = True
os.environ.setdefault("DEFECT_ATLAS_ACCELERATION_MODE", "cpu")
from pathlib import Path
from datetime import datetime

REPO = Path(REPO); BACKEND = REPO / "backend"
_missing = [p for p in (REPO, BACKEND / "app", REPO / "v1-tk-version" / "DefViz-main") if not p.exists()]
if not _missing:
    pass
else:
    # supernames.py accepts QuantificationAnalyzer-main as an alternative
    alt = REPO / "v1-tk-version" / "QuantificationAnalyzer-main"
    _missing = [p for p in _missing if not (p.name == "DefViz-main" and alt.exists())]
if _missing:
    raise RuntimeError("Defect Atlas tree incomplete, missing: " + ", ".join(map(str, _missing))
                       + "\nThe whole REPO must stay intact (supernames.py needs v1-tk-version).")
sys.path.insert(0, str(BACKEND))

import numpy as np, pandas as pd, psutil
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tifffile
import pydantic
if int(pydantic.VERSION.split(".")[0]) >= 2:
    raise RuntimeError(f"pydantic {pydantic.VERSION} found; backend needs 1.x. Use the 'defect-atlas' kernel.")

warnings.filterwarnings("ignore", category=FutureWarning)
from app.domain.models import PipelineConfig
from app.domain.services.pipeline_executor import PipelineExecutor
from app.domain.services.classification import classify_defects, ClassificationConfig
from app.domain.services.shape_classifier import classify_shape
from app.domain.services.projections import compute_projections
from app.services.analytics_service import _volume_key_figures
from app.services.porosity_report_service import compute_direction_report
from app.core.runtime_acceleration import detect_runtime_acceleration
try:
    from app.routers.study import _covariance_ratio_to_axis_ratio as _cov_to_axis_ratio
    _C2A_SRC = "imported from app.routers.study"
except Exception as exc:                                   # router import is optional
    def _cov_to_axis_ratio(value):                         # EXACT copy of study.py:594-597
        if value is None or value <= 0:
            return None
        return math.sqrt(1.0 / value)
    _C2A_SRC = f"replicated from app/routers/study.py:594 (router import failed: {type(exc).__name__})"

def _code_version():
    h = hashlib.md5()
    for p in sorted((BACKEND / "app").rglob("*.py")):
        h.update(p.read_bytes())
    ver = (REPO / "VERSION").read_text().strip() if (REPO / "VERSION").exists() else "?"
    return {"defect_atlas_VERSION": ver, "backend_sources_md5": h.hexdigest()[:12]}
CODE_VERSION = _code_version()

RES_NM = VOXEL_SIZE_MM * 1e6
VOX_UM = RES_NM / 1000.0
PORES_PATH, MASK_PATH = Path(PORES_VOLUME_PATH), Path(MASK_VOLUME_PATH)
OUT = Path(OUT_DIR) if OUT_DIR else PORES_PATH.parent / "pore_analysis"
for sub in ("", "_raw", "figures"): (OUT / sub).mkdir(parents=True, exist_ok=True)

print("python      :", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| pydantic", pydantic.VERSION)
print("backend     :", BACKEND, CODE_VERSION)
print("acceleration:", detect_runtime_acceleration().selected_mode, "(env DEFECT_ATLAS_ACCELERATION_MODE=%s)" % os.environ["DEFECT_ATLAS_ACCELERATION_MODE"])
print("axis ratio  :", _C2A_SRC)
print("voxel size  : %.6g mm = %.6g µm = %.6g nm" % (VOXEL_SIZE_MM, VOX_UM, RES_NM))
print("OUT_DIR     :", OUT)
print("RAM         : %.0f GB available of %.0f GB" % (psutil.virtual_memory().available/2**30, psutil.virtual_memory().total/2**30),
      "| free disk at OUT: %.0f GB" % (psutil.disk_usage(str(OUT)).free/2**30))

python      : 3.11.16 | numpy 1.26.4 | pandas 2.2.3 | pydantic 1.10.17
backend     : /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/10_code/Defect Atlas/backend {'defect_atlas_VERSION': '1.0.0', 'backend_sources_md5': 'f4bb1e073e1b'}
acceleration: cpu (env DEFECT_ATLAS_ACCELERATION_MODE=cpu)
axis ratio  : imported from app.routers.study
voxel size  : 0.025 mm = 25 µm = 25000 nm
OUT_DIR     : /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_01_1/onlypores files/pore_analysis
RAM         : 221 GB available of 249 GB | free disk at OUT: 5431 GB


### 2b. Helpers (thin wrappers around the backend)

In [8]:
LIST_COLS = ("touching_border_faces", "original_labels", "merge_rules")

def build_config(pores_raw, mask_raw, dims_xyz, res_nm, **over):
    p = dict(connectivity=CONNECTIVITY, min_size_voxels=MIN_SIZE_VOXELS, max_size_voxels=MAX_SIZE_VOXELS,
             enable_merging=ENABLE_MERGING, enable_lof_classification=ENABLE_LOF,
             lof_volume_threshold=LOF_VOLUME_THRESHOLD_VOX, lof_elongation_threshold=LOF_ELONGATION_THRESHOLD,
             lof_distance_threshold=LOF_DISTANCE_THRESHOLD_VOX)
    p.update(over)
    return PipelineConfig(
        pores_volume_path=str(pores_raw), mask_volume_path=str(mask_raw), dimensions=tuple(dims_xyz),
        resolution_nm=float(res_nm), data_type="uint8", mask_data_type="uint8", endian="little", header_offset=0,
        calculate_shortest_distance=True, properties_to_compute=None,
        save_csv_excel=False, save_histograms=False, save_scatter_plots=False, save_pore_rotation_gifs=False,
        save_intermediate_results=False, save_distance_map=False, **p)

def defects_to_table(defects):
    """DefectCollection.to_property_dict_list() = label/type/ranking + all props + merge provenance
    (PipelineExecutor._defects_to_dataframe omits type/ranking/merge info, so it is not used)."""
    df = pd.DataFrame(defects.to_property_dict_list())
    df = df.drop(columns=[c for c in ("coords",) if c in df.columns])       # large array column
    for c in LIST_COLS:
        if c in df.columns:
            df[c] = df[c].map(lambda v: "" if v is None or (isinstance(v, float) and v != v) else
                              (",".join(map(str, v)) if isinstance(v, (list, tuple, set)) else
                               json.dumps(v) if isinstance(v, dict) else str(v)))
    return df

def run_backend(pores_raw, mask_raw, dims_xyz, res_nm, progress=None, **over):
    cfg = build_config(pores_raw, mask_raw, dims_xyz, res_nm, **over)
    errors = cfg.validate()
    if errors:
        raise ValueError("PipelineConfig.validate() errors:\n  - " + "\n  - ".join(errors))
    _volume, defects = PipelineExecutor().run_pipeline(cfg, progress_callback=progress)
    df = defects_to_table(defects)
    del _volume, defects
    return df

BORDER_COLS = ["is_boundary_touching", "is_internal_defect", "touching_border_faces",
               "border_touch_voxel_count", "border_touch_fraction", "border_classification"]

def enrich_table(df, res_nm, lof=None, cpx_threshold=None):
    """Border check + classify_defects + shape class. Returns a new DataFrame."""
    lof = lof or dict(volume=LOF_VOLUME_THRESHOLD_VOX, elongation=LOF_ELONGATION_THRESHOLD, distance=LOF_DISTANCE_THRESHOLD_VOX)
    cpx_threshold = CPX_THRESHOLD if cpx_threshold is None else cpx_threshold
    miss = [c for c in BORDER_COLS if c not in df.columns]
    if miss:   # run_pipeline -> label_volume fills these via _compute_border_props; coords are gone by now
        raise RuntimeError(f"border columns missing from pipeline output: {miss}")
    if "original_labels" in df.columns:
        clusters = set(df.loc[df["original_labels"].astype(str) != "", "label"].astype(int))
    else:
        clusters = set()
    cfg = ClassificationConfig(
        volume_threshold_voxels=lof["volume"] if ENABLE_LOF else float("inf"),
        elongation_threshold=lof["elongation"], distance_threshold_voxels=lof["distance"])
    out = classify_defects(df, cfg, cluster_labels=clusters, resolution_nm=res_nm).df.copy()
    # shape class: Cov_* are covariance ratios in (0, 1]; classify_shape wants axis ratios (>= 1): sqrt(1/ratio)
    out["elong_axis_ratio"] = out["Cov_Elongation"].map(_cov_to_axis_ratio)
    out["flat_axis_ratio"]  = out["Cov_Flatness"].map(_cov_to_axis_ratio)
    e, f, c = out["elong_axis_ratio"].astype(float), out["flat_axis_ratio"].astype(float), out["cpx_factor"].astype(float)
    out["shape_class"] = [classify_shape(a, b, k, cpx_threshold) for a, b, k in zip(e, f, c)]
    out["shape_class_no_cpx"] = [classify_shape(a, b, None) for a, b in zip(e, f)]   # elongation/flatness only
    out["Equivalent Diameter (µm)"] = 2.0 * out["Equivalent Radius (µm)"]
    out["Volume3D (mm³)"] = out["Volume3D (µm³)"] * 1e-9
    return out

def check_units(df, res_nm, rtol=1e-6):
    """Physical-unit sanity checks (asserts). Returns a report DataFrame."""
    res_um = res_nm / 1000.0
    rows = []
    def chk(name, got, expect):
        got, expect = np.asarray(got, float), np.asarray(expect, float)
        ok = np.isfinite(got) & np.isfinite(expect)
        err = float(np.max(np.abs(got[ok] - expect[ok]) / np.maximum(np.abs(expect[ok]), 1e-12))) if ok.any() else np.nan
        rows.append((name, err, bool(err <= rtol) if ok.any() else None))
    chk("Volume3D (µm³) == volume_voxels·(res_nm/1000)³", df["Volume3D (µm³)"], df["volume_voxels"] * res_um**3)
    chk("num_pixels == volume_voxels", df["num_pixels"], df["volume_voxels"])
    chk("Equivalent Radius (µm) == (3V/4π)^(1/3)", df["Equivalent Radius (µm)"], (3 * df["Volume3D (µm³)"] / (4 * math.pi)) ** (1/3))
    for k in ("shortest", "mean", "centroid"):
        v = df[f"{k}_distance_to_surface_vox"]
        chk(f"{k}_distance_to_surface_um == vox·res_um", df[f"{k}_distance_to_surface_um"], v * res_um)
        chk(f"{k}_distance_to_surface (legacy) == vox·res_nm  [legacy unit = nm]", df[f"{k}_distance_to_surface"], v * res_nm)
        if f"{k}_distance_to_surface_nm" in df:
            chk(f"{k}_distance_to_surface_nm == vox·res_nm", df[f"{k}_distance_to_surface_nm"], v * res_nm)
    rep = pd.DataFrame(rows, columns=["check", "max_rel_err", "passed"])
    bad = rep[rep["passed"] == False]
    assert bad.empty, "unit checks failed:\n" + bad.to_string()
    return rep
print("helpers defined")

helpers defined


## 3. Load and validate inputs, detect polarity, convert TIFF → RAW

The backend convention is **pores = 255 / material = 0** (labelling binarises `volume > 0`; the EDT is taken on the mask). Polarity is not universal, so for each of the 4 polarity combinations we measure the fraction of pore voxels that fall *outside* the material mask (on a strided sample of slices); the best combination wins (ties prefer "no inversion"). The value histogram of the raw files is shown too. Override with `INVERT_PORES` / `INVERT_MASK` if needed.

**Walls:** `onlypores()` removes the front/back walls from the *pores* only; the saved sample mask still contains material in those slices. The notebook therefore keeps only `front <= z < back` (read from the `_report.txt`, or passed via `WALLS`) for both volumes, so pores, mask (VVF denominator), border flags and distances all refer to the same wall-free box. Pores touching the new z faces are flagged as border. The VVF in the onlypores report (denominator with walls) (denominator includes the walls) will therefore be slightly *lower* than the VVF computed here.

In [9]:
def _open(path):
    t = tifffile.TiffFile(path); s = t.series[0]
    if len(s.shape) != 3: raise ValueError(f"{path}: expected a 3-D TIFF stack, got shape {s.shape}")
    if len(t.pages) != s.shape[0]: raise ValueError(f"{path}: {len(t.pages)} pages but series shape {s.shape}; unsupported layout")
    return t, s

for p in (PORES_PATH, MASK_PATH) + ((Path(GRAY_VOLUME_PATH),) if GRAY_VOLUME_PATH else ()):
    if not p.is_file(): raise FileNotFoundError(f"input volume not found: {p}")

tp, sp = _open(PORES_PATH); tm, sm = _open(MASK_PATH)
if sp.shape != sm.shape: raise ValueError(f"pores {sp.shape} and mask {sm.shape} shapes differ")
if GRAY_VOLUME_PATH:
    tg, sg = _open(Path(GRAY_VOLUME_PATH))
    if sg.shape != sp.shape: raise ValueError(f"gray {sg.shape} differs from pores {sp.shape}")
    tg.close()
FULL_ZYX = tuple(sp.shape)
print("pores TIFF:", FULL_ZYX, sp.dtype, "| mask TIFF:", tuple(sm.shape), sm.dtype)

if CROP is None:
    (z0, z1), (y0, y1), (x0, x1) = [(0, n) for n in FULL_ZYX]
else:
    (z0, z1), (y0, y1), (x0, x1) = CROP
    assert 0 <= z0 < z1 <= FULL_ZYX[0] and 0 <= y0 < y1 <= FULL_ZYX[1] and 0 <= x0 < x1 <= FULL_ZYX[2], "CROP out of range"

# ---- front/back wall exclusion. onlypores() only forces the wall slices to "no pore"; the saved sample mask still
# ---- contains material there, so without this the VVF denominator would include the walls. Keep front <= z < back.
import re
WALL_FRONT = WALL_BACK = None
if WALLS == "auto":
    rep_f = PORES_PATH.parent / PORES_PATH.name.replace("_onlypores.tif", "_report.txt")
    if not rep_f.exists():
        raise FileNotFoundError(f"WALLS='auto' but {rep_f} not found; pass WALLS=(front, back) or WALLS=None")
    txt = rep_f.read_text()
    mf, mb = re.search(r"Front wall slice:\s*(\d+)", txt), re.search(r"Back wall slice:\s*(\d+)", txt)
    if not (mf and mb): raise ValueError(f"could not parse wall slices from {rep_f}")
    WALL_FRONT, WALL_BACK = int(mf.group(1)), int(mb.group(1))
elif WALLS is not None:
    WALL_FRONT, WALL_BACK = map(int, WALLS)
if WALL_FRONT is not None:
    lo = max(WALL_FRONT, 0)                                        # onlypores(): wall applied only if > 0
    hi = WALL_BACK if 0 < WALL_BACK < FULL_ZYX[0] else FULL_ZYX[0]
    z0, z1 = max(z0, lo), min(z1, hi)
    assert z1 > z0, "wall exclusion leaves no slices"
    print(f"walls: front={WALL_FRONT}, back={WALL_BACK} -> analysing z in [{z0}, {z1}) of {FULL_ZYX[0]} "
          f"(excluded {z0} front + {FULL_ZYX[0]-z1} back slices from pores AND mask)")
else:
    print("walls: no wall exclusion")
Z, Y, X = z1 - z0, y1 - y0, x1 - x0
DIMS_XYZ = (X, Y, Z)
N_VOX = X * Y * Z
print(f"working box (Z,Y,X) = ({Z},{Y},{X})  -> DIMENSIONS (X,Y,Z) = {DIMS_XYZ}; {N_VOX/1e6:.1f} Mvoxels; "
      f"box volume = {N_VOX*VOX_UM**3*1e-9:.3f} mm³")
need_gb = N_VOX * 10 / 2**30
print("estimated peak backend memory ≈ %.1f GB (available %.0f GB)" % (need_gb, psutil.virtual_memory().available/2**30))
if need_gb > psutil.virtual_memory().available / 2**30 * 0.8:
    warnings.warn("estimated backend memory is close to / above available RAM: use CROP or expect swapping")

# ---- value histograms + polarity on a strided sample of pages
idx = np.unique(np.linspace(z0, z1 - 1, min(Z, 16)).astype(int))
cnt_p, cnt_m = {}, {}
combos = {(ip, im): 0 for ip in (False, True) for im in (False, True)}   # pore voxels outside mask
ptot = {k: 0 for k in combos}; npx = 0
for i in idx:
    a = tp.pages[int(i)].asarray()[y0:y1, x0:x1]; b = tm.pages[int(i)].asarray()[y0:y1, x0:x1]
    for arr, c in ((a, cnt_p), (b, cnt_m)):
        u, n = np.unique(arr, return_counts=True)
        for uu, nn in zip(u.tolist(), n.tolist()): c[uu] = c.get(uu, 0) + nn
    npx += a.size
    for (ip, im) in combos:
        P = (a == 0) if ip else (a > 0); M = (b == 0) if im else (b > 0)
        combos[(ip, im)] += int((P & ~M).sum()); ptot[(ip, im)] += int(P.sum())
print("\nvalue histogram (sampled pages):\n  pores:", dict(sorted(cnt_p.items())[:8]), "\n  mask :", dict(sorted(cnt_m.items())[:8]))
rows = []
for (ip, im), out in combos.items():
    tot = ptot[(ip, im)]
    rows.append(dict(invert_pores=ip, invert_mask=im, pore_frac_of_box=tot / npx, pores_outside_mask_frac=(out / tot) if tot else np.nan))
pol = pd.DataFrame(rows); display(pol)
best = pol.sort_values(["pores_outside_mask_frac", "invert_pores", "invert_mask"]).iloc[0]
INV_P = bool(best.invert_pores) if INVERT_PORES is None else bool(INVERT_PORES)
INV_M = bool(best.invert_mask)  if INVERT_MASK  is None else bool(INVERT_MASK)
print(f"POLARITY -> invert_pores={INV_P} ({'auto' if INVERT_PORES is None else 'forced'}), "
      f"invert_mask={INV_M} ({'auto' if INVERT_MASK is None else 'forced'})  "
      f"[pores are {'0' if INV_P else 'non-zero'} in the file, material is {'0' if INV_M else 'non-zero'}]")
tp.close(); tm.close()

pores TIFF: (208, 3405, 1703) uint8 | mask TIFF: (208, 3405, 1703) uint8
walls: front=8, back=193 -> analysing z in [8, 193) of 208 (excluded 8 front + 15 back slices from pores AND mask)
working box (Z,Y,X) = (185,3405,1703)  -> DIMENSIONS (X,Y,Z) = (1703, 3405, 185); 1072.8 Mvoxels; box volume = 16761.911 mm³
estimated peak backend memory ≈ 10.0 GB (available 221 GB)

value histogram (sampled pages):
  pores: {0: 90881437, 255: 1898003} 
  mask : {0: 4909358, 255: 87870082}


,invert_pores,invert_mask,pore_frac_of_box,pores_outside_mask_frac
0,False,False,0.020457,0.000000
1,False,True,0.020457,1.000000
2,True,False,0.979543,0.054019
3,True,True,0.979543,0.945981


POLARITY -> invert_pores=False (auto), invert_mask=False (auto)  [pores are non-zero in the file, material is non-zero]


In [10]:
def convert_to_raw(pores_tif, mask_tif, raw_dir, crop_box, inv_p, inv_m):
    """Stream TIFF pages -> normalised uint8 RAW (0/255) and count voxels on the way."""
    (z0, z1), (y0, y1), (x0, x1) = crop_box
    pr, mr = raw_dir / "pores.raw", raw_dir / "mask.raw"
    cnt = dict(pore_voxels=0, mask_voxels=0, pores_outside_mask=0)
    with tifffile.TiffFile(pores_tif) as tp, tifffile.TiffFile(mask_tif) as tm, open(pr, "wb") as fp, open(mr, "wb") as fm:
        for i in tqdm(range(z0, z1), desc="TIFF -> RAW"):
            a = tp.pages[i].asarray()[y0:y1, x0:x1]; b = tm.pages[i].asarray()[y0:y1, x0:x1]
            P = (a == 0) if inv_p else (a > 0); M = (b == 0) if inv_m else (b > 0)
            cnt["pore_voxels"] += int(P.sum()); cnt["mask_voxels"] += int(M.sum()); cnt["pores_outside_mask"] += int((P & ~M).sum())
            fp.write((P.astype(np.uint8) * 255).tobytes()); fm.write((M.astype(np.uint8) * 255).tobytes())
    return pr, mr, cnt

def _fileid(p): s = Path(p).stat(); return [str(p), s.st_size, int(s.st_mtime)]
RAW_DIR = OUT / "_raw"
raw_key = dict(pores=_fileid(PORES_PATH), mask=_fileid(MASK_PATH), box=[[z0, z1], [y0, y1], [x0, x1]], inv=[INV_P, INV_M])
meta_f = RAW_DIR / "raw_meta.json"
PORES_RAW, MASK_RAW = RAW_DIR / "pores.raw", RAW_DIR / "mask.raw"
cached = None
if USE_CACHE and meta_f.exists() and PORES_RAW.exists() and MASK_RAW.exists():
    m = json.loads(meta_f.read_text())
    if m["key"] == raw_key and PORES_RAW.stat().st_size == N_VOX == MASK_RAW.stat().st_size: cached = m["counts"]
if cached is None:
    t0 = time.time(); PORES_RAW, MASK_RAW, COUNTS = convert_to_raw(PORES_PATH, MASK_PATH, RAW_DIR, ((z0, z1), (y0, y1), (x0, x1)), INV_P, INV_M)
    meta_f.write_text(json.dumps(dict(key=raw_key, counts=COUNTS))); print("converted in %.0fs" % (time.time() - t0))
else:
    COUNTS = cached; print("RAW cache reused")

# expected bytes = X*Y*Z*itemsize + header(0)
for p in (PORES_RAW, MASK_RAW):
    assert p.stat().st_size == N_VOX, f"{p}: {p.stat().st_size} bytes, expected {N_VOX}"
MASK_VOX, PORE_VOX_TOTAL = COUNTS["mask_voxels"], COUNTS["pore_voxels"]
print(f"\nRAW sizes OK ({N_VOX} bytes each)")
print(f"pore voxels     : {PORE_VOX_TOTAL:,}  ({100*PORE_VOX_TOTAL/N_VOX:.3f}% of box)")
print(f"material voxels : {MASK_VOX:,}  ({100*MASK_VOX/N_VOX:.2f}% of box)")
print(f"pore voxels outside mask: {COUNTS['pores_outside_mask']:,}")
print(f"VVF (pore voxels / material voxels) = {100*PORE_VOX_TOTAL/MASK_VOX:.4f} %")
rep = PORES_PATH.parent / (PORES_PATH.name.replace("_onlypores.tif", "_report.txt"))
if rep.exists() and CROP is None and WALL_FRONT is None:
    import re
    m_ = re.search(r"Void volumetric fraction.*?\(([\d.]+)%\)", rep.read_text())
    print(f"onlypores report.txt says VVF = {m_.group(1)} %  (cross-check)" if m_ else "report.txt found, VVF line not parsed")

# mid-slice preview from the normalised RAW files
pm = np.memmap(PORES_RAW, np.uint8, "r", shape=(Z, Y, X)); mm = np.memmap(MASK_RAW, np.uint8, "r", shape=(Z, Y, X))
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
k = Z // 2
ax[0].imshow(pm[k], cmap="gray"); ax[0].set_title(f"pores (z={k})")
ax[1].imshow(mm[k], cmap="gray"); ax[1].set_title(f"material mask (z={k})")
ov = np.zeros((Y, X, 3), np.float32); ov[..., 1] = mm[k] > 0; ov[..., 0] = pm[k] > 0; ov[..., 2] = 0
ax[2].imshow(ov); ax[2].set_title("overlay: red = pores, green = material")
for a_ in ax: a_.set_xlabel("x (voxel)"); a_.set_ylabel("y (voxel)")
plt.tight_layout(); plt.show()

RAW cache reused

RAW sizes OK (1072762275 bytes each)
pore voxels     : 21,387,636  (1.994% of box)
material voxels : 1,016,398,666  (94.75% of box)
pore voxels outside mask: 0
VVF (pore voxels / material voxels) = 2.1043 %


/tmp/ipykernel_1124717/1429970079.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Run the backend pipeline (label → all properties → optional merge)

In [11]:
cfg = build_config(PORES_RAW, MASK_RAW, DIMS_XYZ, RES_NM)
errs = cfg.validate()
print("PipelineConfig.validate() ->", errs)
assert errs == [], "fix the errors above before running"

run_key = dict(raw=raw_key, res_nm=RES_NM, min=MIN_SIZE_VOXELS, max=MAX_SIZE_VOXELS, conn=CONNECTIVITY,
               merge=ENABLE_MERGING, code=CODE_VERSION)
RAW_TBL, RAW_KEY = OUT / "pores_raw.parquet", OUT / "pores_raw.key.json"
if USE_CACHE and RAW_TBL.exists() and RAW_KEY.exists() and json.loads(RAW_KEY.read_text()) == run_key:
    df_raw = pd.read_parquet(RAW_TBL); RUNTIME_S = None
    print(f"pipeline result loaded from cache: {RAW_TBL}  ({len(df_raw):,} pores)")
else:
    bar = tqdm(total=100, desc="pipeline", bar_format="{l_bar}{bar}| {n:.0f}% {postfix}")
    def progress(msg, pct):
        bar.n = min(100, float(pct)); bar.set_postfix_str(str(msg)[:60]); bar.refresh()
    t0 = time.time()
    df_raw = run_backend(PORES_RAW, MASK_RAW, DIMS_XYZ, RES_NM, progress=progress)
    RUNTIME_S = time.time() - t0; bar.n = 100; bar.refresh(); bar.close()
    df_raw.to_parquet(RAW_TBL); RAW_KEY.write_text(json.dumps(run_key))
    print(f"{len(df_raw):,} pores in {RUNTIME_S:.0f}s; cached to {RAW_TBL}")

PipelineConfig.validate() -> []


pipeline: 100%|██████████| 100% , Pipeline complete                                          


226,030 pores in 562s; cached to /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_01_1/onlypores files/pore_analysis/pores_raw.parquet


## 5. Border properties, classification (Pore / LoF / Cluster / Border) and shape class

* **Border:** `run_pipeline` already fills the six border columns (`PipelineExecutor.label_volume` → `_compute_border_props`, coords are `(z, y, x)`, maxima inclusive); the notebook verifies they exist instead of recomputing.
* **Classification:** `classify_defects` with the thresholds from the parameter cell (priority: LoF → Border → Cluster/Pore).
* **Shape class:** `Cov_Elongation` / `Cov_Flatness` are covariance ratios in (0, 1]; `classify_shape` needs axis ratios (major/minor ≥ 1). They are converted with `_covariance_ratio_to_axis_ratio` = `sqrt(1/ratio)` (**reused from `app/routers/study.py:594`**, see setup output for whether it was imported or replicated). `shape_class` uses `cpx_factor` (surface-area override above `CPX_THRESHOLD`); `shape_class_no_cpx` uses elongation/flatness only.

In [12]:
df = enrich_table(df_raw, RES_NM)
print("type        :", df["type"].value_counts().to_dict())
print("shape_class :", df["shape_class"].value_counts(dropna=False).to_dict())
print("no_cpx class:", df["shape_class_no_cpx"].value_counts(dropna=False).to_dict())
print("border      :", df["border_classification"].value_counts().to_dict())

type        : {'Pore': 209969, 'LoF': 15932, 'Border': 129}
shape_class : {'Complex': 160516, None: 65498, 'Ellipsoidal': 10, 'Disc': 6}
no_cpx class: {'Ellipsoidal': 150523, 'Disc': 39260, 'Rod': 35924, 'Spherical': 320, None: 3}
border      : {'internal': 225821, 'single_face': 207, 'edge': 2}


## 6. Per-pore DataFrame

In [13]:
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
display(df.head())
display(df.describe().T)
print(f"{len(df):,} pores x {df.shape[1]} columns")

,label,type,ranking,is_boundary_touching,is_internal_defect,touching_border_faces,border_touch_voxel_count,border_touch_fraction,border_classification,BoundingBox_Dx (µm),BoundingBox_Dy (µm),BoundingBox_Dz (µm),bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,centroid-0,centroid-1,centroid-2,num_pixels,Covariance_Eigenvalue_1,Covariance_Eigenvalue_2,Covariance_Eigenvalue_3,Covariance_Eigenvector_1_x,Covariance_Eigenvector_1_y,Covariance_Eigenvector_1_z,Covariance_Eigenvector_2_x,Covariance_Eigenvector_2_y,...,shortest_distance_to_surface_um,mean_distance_to_surface_vox,mean_distance_to_surface,mean_distance_to_surface_nm,mean_distance_to_surface_um,centroid_distance_to_surface_vox,centroid_distance_to_surface,centroid_distance_to_surface_nm,centroid_distance_to_surface_um,Area3D (µm²),border_voxel_count,Area3D_capped,Volume3D (µm³),volume_voxels,Ellipsoid_Major_Axis_Length (µm),Ellipsoid_Medium_Axis_Length (µm),Ellipsoid_Minor_Axis_Length (µm),Ellipsoid_Semi_Axis_1 (µm),Ellipsoid_Semi_Axis_2 (µm),Ellipsoid_Semi_Axis_3 (µm),cpx_factor,cpx_factor_capped,Sphericity,Equivalent Radius (µm),elong_axis_ratio,flat_axis_ratio,shape_class,shape_class_no_cpx,Equivalent Diameter (µm),Volume3D (mm³)
0,1,LoF,7,True,False,"y_min,x_max",917,0.001783,edge,14125.0,18925.0,4025.0,17,0,1138,178,757,1703,94.071118,287.281334,1527.142226,514357,2.347891e+07,7.306915e+06,714643.826108,-0.020878,0.999777,0.003074,0.998812,0.020723,...,25.000000,107.735802,2.693395e+06,2.693395e+06,2693.395042,116.691902,2.917298e+06,2.917298e+06,2917.297554,NaN,0,True,8.036828e+09,514357,5402.780898,3014.015286,942.590920,2701.390449,1507.007643,471.295460,NaN,True,NaN,1242.601930,1.792553,3.197586,None,Ellipsoidal,2485.203860,8.036828
1,2,LoF,2,False,True,,0,0.000000,internal,8175.0,12225.0,500.0,106,2914,1311,126,3403,1638,113.954882,3239.884493,1509.633395,80610,9.850391e+06,2.612711e+06,2449.337773,-0.402541,0.915398,0.002668,0.915401,0.402544,...,25.000000,74.966492,1.874162e+06,1.874162e+06,1874.162292,131.000000,3.275000e+06,3.275000e+06,3275.000000,7.269797e+07,183488,False,1.259531e+09,80610,6665.988907,3433.076989,105.114395,3332.994453,1716.538494,52.557198,40.869743,False,0.077583,669.946485,1.941695,32.660389,Complex,Ellipsoidal,1339.892969,1.259531
2,3,LoF,3,False,True,,0,0.000000,internal,9800.0,10300.0,600.0,102,2736,1252,126,3148,1644,114.482534,2941.884269,1429.176128,55681,1.054374e+07,1.683041e+06,2229.638557,-0.578852,0.815360,0.010871,0.815425,0.578851,...,250.000000,166.538193,4.163455e+06,4.163455e+06,4163.454819,198.037872,4.950947e+06,4.950947e+06,4950.946808,4.968386e+07,125590,False,8.700156e+08,55681,6588.494555,2632.304946,95.808941,3294.247278,1316.152473,47.904470,71.874499,False,0.088706,592.215106,2.502937,27.474523,Complex,Ellipsoidal,1184.430212,0.870016
3,4,LoF,24,False,True,,0,0.000000,internal,5400.0,9875.0,900.0,53,90,1194,89,485,1410,71.050918,307.015932,1297.784922,47076,4.735989e+06,1.567740e+06,18252.043714,-0.385113,0.922558,-0.023959,0.921421,0.385833,...,2225.561714,244.894241,6.122356e+06,6.122356e+06,6122.356033,262.619507,6.565488e+06,6.565488e+06,6565.487671,3.643049e+07,88236,False,7.355625e+08,47076,3400.788689,1956.641484,211.120329,1700.394344,978.320742,105.560164,66.539383,False,0.108168,559.985769,1.738075,9.267897,Complex,Ellipsoidal,1119.971538,0.735563
4,5,LoF,19,False,True,,0,0.000000,internal,2950.0,9275.0,1075.0,54,451,1567,97,822,1685,71.839801,594.490448,1627.038747,33396,3.762800e+06,2.677310e+05,22659.408430,-0.087278,0.996184,-0.000565,0.988361,0.086664,...,237.170815,64.057312,1.601433e+06,1.601433e+06,1601.432800,64.124878,1.603122e+06,1.603122e+06,1603.121948,2.724966e+07,66954,False,5.218125e+08,33396,3637.903271,970.386777,282.305825,1818.951636,485.193388,141.152912,57.324111,False,0.115027,499.430763,3.748921,3.437360,Complex,Rod,998.861526,0.521813


,count,mean,std,min,25%,50%,75%,max
label,226030.0,113015.500000,65249.385010,1.000000,56508.250000,113015.500000,169522.750000,2.260300e+05
ranking,226030.0,561.528461,2376.779105,0.000000,0.000000,0.000000,0.000000,1.593200e+04
border_touch_voxel_count,226030.0,0.017564,2.924623,0.000000,0.000000,0.000000,0.000000,1.028000e+03
border_touch_fraction,226030.0,0.000192,0.008292,0.000000,0.000000,0.000000,0.000000,8.888890e-01
BoundingBox_Dx (µm),226030.0,210.292328,220.929553,50.000000,100.000000,150.000000,225.000000,1.412500e+04
...,...,...,...,...,...,...,...,...
Equivalent Radius (µm),226030.0,55.602129,26.697675,32.259526,37.378323,47.093736,64.218958,1.242602e+03
elong_axis_ratio,226030.0,2.230554,1.146776,1.000000,1.530243,1.950571,2.589209,3.208543e+01
flat_axis_ratio,226027.0,263.420680,124269.474279,1.000000,1.407828,1.731880,2.279916,5.908056e+07
Equivalent Diameter (µm),226030.0,111.204258,53.395350,64.519051,74.756647,94.187473,128.437916,2.485204e+03


226,030 pores x 134 columns


In [14]:
EXPECTED = {
 "Size": ["Volume3D (µm³)", "volume_voxels", "num_pixels", "Equivalent Radius (µm)", "Equivalent Diameter (µm)"],
 "Surface": ["Area3D (µm²)", "border_voxel_count", "Area3D_capped", "Sphericity"],
 "Position": ["centroid-0", "centroid-1", "centroid-2"] + [f"bbox-{i}" for i in range(6)] + ["BoundingBox_Dx (µm)", "BoundingBox_Dy (µm)", "BoundingBox_Dz (µm)"],
 "Feret": ["Length3D (µm)", "Width3D (µm)", "Breadth3D (µm)", "Thickness3D (µm)"],
 "Ellipsoid": ["Ellipsoid_Major_Axis_Length (µm)", "Ellipsoid_Medium_Axis_Length (µm)", "Ellipsoid_Minor_Axis_Length (µm)",
               "Ellipsoid_Semi_Axis_1 (µm)", "Ellipsoid_Semi_Axis_2 (µm)", "Ellipsoid_Semi_Axis_3 (µm)"]
               + [f"OrientedBBox_Corner{i}_{a} (µm)" for i in range(1, 9) for a in "xyz"],
 "Tensor shape": [f"{p}_{q}" for p in ("Cov", "Inertia") for q in ("Elongation", "Flatness", "Anisotropy", "elongation_avg", "flatness_avg")]
               + [f"Covariance_Eigenvalue_{i}" for i in (1, 2, 3)] + [f"Inertia_Eigenvalue_{i}" for i in (1, 2, 3)]
               + [f"Covariance_Eigenvector_{i}_{a}" for i in (1, 2, 3) for a in "xyz"] + [f"Inertia_Eigenvector_{i}_{a}" for i in (1, 2, 3) for a in "xyz"],
 "Orientation": ["Phi (degrees)", "Theta (degrees)", "Cov_Phi (degrees)", "Cov_Theta (degrees)", "Inertia_Phi (degrees)", "Inertia_Theta (degrees)"],
 "Complexity": ["cpx_factor", "cpx_factor_capped"],
 "Distance to surface": [f"{k}_distance_to_surface{s}" for k in ("shortest", "mean", "centroid") for s in ("_vox", "", "_nm", "_um")],
 "Border": BORDER_COLS,
 "Classification": ["type", "ranking", "shape_class", "shape_class_no_cpx"],
}
if ENABLE_MERGING: EXPECTED["Merge provenance"] = ["original_labels", "merge_rules"]
rows = []
for grp, cols in EXPECTED.items():
    for c in cols:
        if c not in df.columns: rows.append((grp, c, "MISSING", np.nan))
        else:
            n_nan = int(df[c].isna().sum())
            rows.append((grp, c, "ALL NaN" if n_nan == len(df) and len(df) else "ok", n_nan))
chk = pd.DataFrame(rows, columns=["group", "column", "status", "n_nan"])
print(chk.groupby(["group", "status"]).size().unstack(fill_value=0).to_string())
display(chk[chk.status != "ok"])
extra = [c for c in df.columns if c not in set(sum(EXPECTED.values(), []))]
print("columns present but not in the spec groups:", extra)

# missing-values report (columns with any NaN)
nn = df.isna().sum(); nn = nn[nn > 0].sort_values(ascending=False)
print(f"\ncolumns with NaNs: {len(nn)}"); display((pd.DataFrame({"n_nan": nn, "pct": 100 * nn / len(df)})).head(30))

# capped / skipped-for-large-pore flags stay visible
for c in ("Area3D_capped", "cpx_factor_capped"):
    if c in df: print(f"{c}: {int(df[c].astype(bool).sum())} of {len(df)} pores capped")

status               ok
group                  
Border                6
Classification        4
Complexity            2
Distance to surface  12
Ellipsoid            30
Feret                 4
Orientation           6
Position             12
Size                  5
Surface               4
Tensor shape         34


,group,column,status,n_nan


columns present but not in the spec groups: ['label', 'Elongation', 'Flatness', 'Anisotropy', 'elongation_avg', 'flatness_avg', 'Inertia_Ixx (µm²)', 'Inertia_Ixy (µm²)', 'Inertia_Ixz (µm²)', 'Inertia_Iyy (µm²)', 'Inertia_Iyz (µm²)', 'Inertia_Izz (µm²)', 'elong_axis_ratio', 'flat_axis_ratio', 'Volume3D (mm³)']

columns with NaNs: 18


,n_nan,pct
shape_class,65498,28.977569
cpx_factor,65498,28.977569
Cov_Phi (degrees),21,0.009291
Cov_Theta (degrees),21,0.009291
Phi (degrees),21,0.009291
Theta (degrees),21,0.009291
flatness_avg,3,0.001327
Ellipsoid_Semi_Axis_2 (µm),3,0.001327
flat_axis_ratio,3,0.001327
Ellipsoid_Semi_Axis_3 (µm),3,0.001327


Area3D_capped: 1 of 226030 pores capped
cpx_factor_capped: 2 of 226030 pores capped


### Physical-unit sanity check (mandatory)
`Volume3D (µm³) = volume_voxels · (RES_NM/1000)³`; distances: `*_vox · RES_NM` (**nm**) for the legacy `*_distance_to_surface` columns (their display label in the backend says *nm* – and that is what they are; the `_um` columns are in µm).

In [15]:
units_report = check_units(df, RES_NM)
display(units_report)
print("ALL UNIT CHECKS PASSED  (volume in µm³, legacy distance columns in nm, *_um in µm, *_vox in voxels)")

,check,max_rel_err,passed
0,Volume3D (µm³) == volume_voxels·(res_nm/1000)³,1.186630e-16,True
1,num_pixels == volume_voxels,0.000000e+00,True
2,Equivalent Radius (µm) == (3V/4π)^(1/3),0.000000e+00,True
3,shortest_distance_to_surface_um == vox·res_um,0.000000e+00,True
4,shortest_distance_to_surface (legacy) == vox·r...,0.000000e+00,True
5,shortest_distance_to_surface_nm == vox·res_nm,0.000000e+00,True
6,mean_distance_to_surface_um == vox·res_um,0.000000e+00,True
7,mean_distance_to_surface (legacy) == vox·res_n...,0.000000e+00,True
8,mean_distance_to_surface_nm == vox·res_nm,0.000000e+00,True
9,centroid_distance_to_surface_um == vox·res_um,0.000000e+00,True


ALL UNIT CHECKS PASSED  (volume in µm³, legacy distance columns in nm, *_um in µm, *_vox in voxels)


## 7. Specimen summary

In [16]:
def specimen_summary(df, mask_vox, n_vox, pore_vox_total, res_nm):
    vox_um3 = (res_nm / 1000.0) ** 3
    kf = _volume_key_figures(df)                                  # analytics_service: d = (6V/π)^(1/3)
    pore_um3 = float(df["Volume3D (µm³)"].sum())
    mat_mm3, box_mm3 = mask_vox * vox_um3 * 1e-9, n_vox * vox_um3 * 1e-9
    internal = df[df["is_internal_defect"].astype(bool)]
    s = dict(
        voxel_size_um=res_nm / 1000.0, box_voxels=int(n_vox), box_volume_mm3=box_mm3,
        material_voxels=int(mask_vox), material_volume_mm3=mat_mm3,
        defect_count=int(len(df)),
        porosity_pct=100.0 * pore_um3 * 1e-9 / mat_mm3,                 # study.py / routers/pipeline.py: pore volume / mask volume
        porosity_box_pct=100.0 * pore_um3 * 1e-9 / box_mm3,
        porosity_internal_only_pct=100.0 * float(internal["Volume3D (µm³)"].sum()) * 1e-9 / mat_mm3,
        defect_density_per_mm3=len(df) / mat_mm3,
        total_defect_volume_um3=pore_um3, total_defect_volume_mm3=pore_um3 * 1e-9,
        largest_volume_um3=kf.get("largest_volume_um3"), largest_eq_diameter_um=kf.get("largest_eq_diameter_um"),
        median_eq_diameter_um=kf.get("median_eq_diameter_um"),
        pore_voxels_in_table=int(df["volume_voxels"].sum()), pore_voxels_in_volume=int(pore_vox_total),
        counts_by_type=df["type"].value_counts().to_dict(),
        counts_by_shape_class=df["shape_class"].value_counts(dropna=False).rename(lambda k: str(k)).to_dict(),
        counts_by_shape_class_no_cpx=df["shape_class_no_cpx"].value_counts(dropna=False).rename(lambda k: str(k)).to_dict(),
        counts_by_border_class=df["border_classification"].value_counts().to_dict(),
        n_boundary_touching=int(df["is_boundary_touching"].astype(bool).sum()),
        n_area_capped=int(df["Area3D_capped"].astype(bool).sum()) if "Area3D_capped" in df else None,
        n_cpx_capped=int(df["cpx_factor_capped"].astype(bool).sum()) if "cpx_factor_capped" in df else None,
    )
    return s

SUMMARY = specimen_summary(df, MASK_VOX, N_VOX, PORE_VOX_TOTAL, RES_NM)
display(pd.Series({k: v for k, v in SUMMARY.items() if not isinstance(v, dict)}, name="value").to_frame())
for k in ("counts_by_type", "counts_by_shape_class", "counts_by_shape_class_no_cpx", "counts_by_border_class"):
    print(k, SUMMARY[k])
d_vox = SUMMARY["pore_voxels_in_volume"] - SUMMARY["pore_voxels_in_table"]
print(f"\npore voxels in volume − in table = {d_vox:,}  (non-zero only if MIN_SIZE/MAX_SIZE filtering removed pores)")

,value
voxel_size_um,2.500000e+01
box_voxels,1.072762e+09
box_volume_mm3,1.676191e+04
material_voxels,1.016399e+09
material_volume_mm3,1.588123e+04
defect_count,2.260300e+05
porosity_pct,2.104257e+00
porosity_box_pct,1.993698e+00
porosity_internal_only_pct,2.046651e+00
defect_density_per_mm3,1.423253e+01


counts_by_type {'Pore': 209969, 'LoF': 15932, 'Border': 129}
counts_by_shape_class {'Complex': 160516, 'None': 65498, 'Ellipsoidal': 10, 'Disc': 6}
counts_by_shape_class_no_cpx {'Ellipsoidal': 150523, 'Disc': 39260, 'Rod': 35924, 'Spherical': 320, 'None': 3}
counts_by_border_class {'internal': 225821, 'single_face': 207, 'edge': 2}

pore voxels in volume − in table = 0  (non-zero only if MIN_SIZE/MAX_SIZE filtering removed pores)


## 8. Plots

In [17]:
FIG = OUT / "figures"
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
TYPE_COL = {"Pore": "#1f77b4", "LoF": "#d62728", "Cluster": "#2ca02c", "Border": "#ff7f0e"}

# data-quality flags (shown, not hidden): covariance ratios should lie in (0, 1]; sphericity should be <= 1
n_cov_bad = int(((df["Cov_Elongation"] <= 0) | (df["Cov_Flatness"] <= 0)).sum()); n_cov_nan = int(df["Cov_Elongation"].isna().sum())
n_sph_gt1 = int((df["Sphericity"] > 1.0 + 1e-9).sum()); n_1vox = int((df["volume_voxels"] <= 1).sum())
print(f"Cov_Elongation/Cov_Flatness <= 0 (numerically degenerate, no shape class): {n_cov_bad}; NaN: {n_cov_nan}")
print(f"Sphericity > 1 (voxelised area of tiny pores): {n_sph_gt1}; pores with <= 1 voxel: {n_1vox}  (use MIN_SIZE_VOXELS to drop them)")
SUMMARY.update(n_cov_ratio_nonpositive=n_cov_bad, n_cov_ratio_nan=n_cov_nan, n_sphericity_gt1=n_sph_gt1, n_single_voxel_pores=n_1vox)
vol = df["Volume3D (µm³)"]; vol = vol[vol > 0]
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
ax[0, 0].hist(vol, bins=np.logspace(np.log10(vol.min()), np.log10(vol.max()), 60)); ax[0, 0].set_xscale("log"); ax[0, 0].set_yscale("log")
ax[0, 0].set_xlabel("Volume (µm³)"); ax[0, 0].set_ylabel("pore count"); ax[0, 0].set_title("Volume")
ed = df["Equivalent Diameter (µm)"]; ax[0, 1].hist(ed.dropna(), bins=60); ax[0, 1].set_yscale("log")
ax[0, 1].set_xlabel("Equivalent diameter (µm)"); ax[0, 1].set_ylabel("pore count"); ax[0, 1].set_title("Equivalent diameter")
ax[1, 0].hist(df["Sphericity"].dropna(), bins=60); ax[1, 0].set_xlabel("Sphericity (-)"); ax[1, 0].set_ylabel("pore count"); ax[1, 0].set_title("Sphericity")
ax[1, 1].hist(df["Cov_Elongation"].dropna(), bins=60, range=(0, 1), alpha=.7, label="Cov_Elongation (λ2/λ1)")
ax[1, 1].hist(df["Cov_Flatness"].dropna(), bins=60, range=(0, 1), alpha=.7, label="Cov_Flatness (λ3/λ2)")
ax[1, 1].set_xlabel("covariance ratio (-)"); ax[1, 1].set_ylabel("pore count"); ax[1, 1].legend(); ax[1, 1].set_title("Elongation / flatness")
plt.tight_layout(); save(fig, "hist_volume_diameter_sphericity_elongation")

fig, ax = plt.subplots(figsize=(8, 6))
for t, g in df.groupby("type"):
    ax.scatter(g["Volume3D (µm³)"], g["Sphericity"], s=6, alpha=.4, c=TYPE_COL.get(t, "grey"), label=f"{t} ({len(g):,})", rasterized=True)
ax.set_xscale("log"); ax.set_xlabel("Volume (µm³)"); ax.set_ylabel("Sphericity (-)"); ax.legend(markerscale=3); ax.set_title("Volume vs sphericity by type")
save(fig, "scatter_volume_sphericity_by_type")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a_, col, rng in ((ax[0], "Phi", (-180, 180)), (ax[1], "Theta", (0, 180))):
    for pre, lab in (("", "main"), ("Cov_", "cov"), ("Inertia_", "inertia")):
        v = df[f"{pre}{col} (degrees)"].dropna()
        a_.hist(v, bins=60, histtype="step", label=f"{lab} (n={len(v):,})")
    a_.set_xlabel(f"{col} (degrees)"); a_.set_ylabel("pore count"); a_.legend(); a_.set_title(f"Orientation: {col}")
plt.tight_layout(); save(fig, "hist_phi_theta")

Cov_Elongation/Cov_Flatness <= 0 (numerically degenerate, no shape class): 3; NaN: 0
Sphericity > 1 (voxelised area of tiny pores): 23462; pores with <= 1 voxel: 0  (use MIN_SIZE_VOXELS to drop them)


/tmp/ipykernel_1124717/3486177051.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()
/tmp/ipykernel_1124717/3486177051.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()


In [18]:
# per-slice porosity (backend: porosity_report_service.compute_direction_report), RAW files are already 0/255 => invert=False
SLICE = {}
if RUN_SLICE_PROFILE:
    for axis in SLICE_AXES:
        f = OUT / f"slice_profile_{axis}.csv"; kf_ = OUT / f"slice_profile_{axis}.key.json"
        if USE_CACHE and f.exists() and kf_.exists() and json.loads(kf_.read_text()) == raw_key:
            SLICE[axis] = pd.read_csv(f); continue
        bar = tqdm(total=1, desc=f"slices {axis}")
        def cb(done, total): bar.total = total; bar.n = done; bar.refresh()
        SLICE[axis] = pd.DataFrame(compute_direction_report(pm, mm, axis, VOX_UM, False, False, cb)); bar.close()
        SLICE[axis].to_csv(f, index=False); kf_.write_text(json.dumps(raw_key))
    fig, ax = plt.subplots(1, len(SLICE), figsize=(5.5 * len(SLICE), 4), squeeze=False)
    for a_, (axis, d) in zip(ax[0], SLICE.items()):
        a_.plot(d["slice_um"], d["porosity_percent"]); a_.set_xlabel(f"{axis} position (µm)"); a_.set_ylabel("porosity, area % of material (%)")
        a_.set_title(f"Porosity profile along {axis.upper()}")
    plt.tight_layout(); save(fig, "porosity_profile_slices")
    print({a: dict(mean_pct=float(d["porosity_percent"].mean()), max_pct=float(d["porosity_percent"].max())) for a, d in SLICE.items()})
else:
    print("RUN_SLICE_PROFILE=False")

# porosity vs Z from the pore table (centroid slab volume / material volume per slab), independent cross-check
zc = df["centroid-0"].to_numpy(float); zv = df["Volume3D (µm³)"].to_numpy(float)
nb = min(40, Z); edges = np.linspace(0, Z, nb + 1)
mat_z = np.array([np.count_nonzero(mm[int(a):int(b)]) for a, b in zip(edges[:-1], edges[1:])], float) * VOX_UM**3
slab_por = 100 * np.histogram(zc, edges, weights=zv)[0] / np.maximum(mat_z, 1)
fig, ax = plt.subplots(figsize=(7, 4)); ax.step(edges[:-1] * VOX_UM, slab_por, where="post")
ax.set_xlabel("z position (µm)"); ax.set_ylabel("slab porosity (%)"); ax.set_title("Porosity vs Z (pore centroid slabs)")
save(fig, "porosity_vs_z_slabs")

{'z': {'mean_pct': 2.103338225753929, 'max_pct': 12.4773024173153}, 'y': {'mean_pct': 2.0982554607121595, 'max_pct': 3.2493146362148195}, 'x': {'mean_pct': 2.3467044910447905, 'max_pct': 99.61089494163424}}


/tmp/ipykernel_1124717/3486177051.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()


In [19]:
if RUN_PROJECTION:
    pr = compute_projections(pm)           # streamed along Z in slabs
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    im0 = ax[0].imshow(pr.max_pores, cmap="gray_r"); ax[0].set_title("Z max projection (any pore in column)")
    im1 = ax[1].imshow(pr.avg_pores, cmap="viridis"); ax[1].set_title("Z average projection"); plt.colorbar(im1, ax=ax[1], label="% of Z height that is pore")
    for a_ in ax: a_.set_xlabel("x (voxel)"); a_.set_ylabel("y (voxel)")
    plt.tight_layout(); save(fig, "projection_z")

/tmp/ipykernel_1124717/3486177051.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); plt.show()


## 9. Export

In [20]:
cols_out = [c for c in df.columns if c != "coords"]
df[cols_out].to_csv(OUT / "pores.csv", index=False)
df[cols_out].to_parquet(OUT / "pores.parquet", index=False)
(OUT / "specimen_summary.json").write_text(json.dumps(SUMMARY, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
run_config = dict(
    created=datetime.now().isoformat(timespec="seconds"), code_version=CODE_VERSION, repo=str(REPO),
    kernel_python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, pydantic=pydantic.VERSION,
    inputs=dict(walls=[WALL_FRONT, WALL_BACK], analysed_z_range=[z0, z1], pores=str(PORES_PATH), mask=str(MASK_PATH), gray=GRAY_VOLUME_PATH, crop=CROP, full_shape_zyx=list(FULL_ZYX),
                working_shape_zyx=[Z, Y, X]),
    polarity=dict(invert_pores=INV_P, invert_mask=INV_M, auto_pores=INVERT_PORES is None, auto_mask=INVERT_MASK is None),
    voxel_size_mm=VOXEL_SIZE_MM, resolution_nm=RES_NM,
    pipeline=dict(connectivity=CONNECTIVITY, min_size_voxels=MIN_SIZE_VOXELS, max_size_voxels=MAX_SIZE_VOXELS,
                  enable_merging=ENABLE_MERGING, enable_lof=ENABLE_LOF, calculate_shortest_distance=True, properties_to_compute=None),
    classification=dict(volume_threshold_voxels=LOF_VOLUME_THRESHOLD_VOX, elongation_threshold=LOF_ELONGATION_THRESHOLD,
                        distance_threshold_voxels=LOF_DISTANCE_THRESHOLD_VOX, cpx_threshold=CPX_THRESHOLD,
                        axis_ratio_conversion="sqrt(1/cov_ratio), " + _C2A_SRC),
    acceleration=os.environ["DEFECT_ATLAS_ACCELERATION_MODE"], pipeline_runtime_s=RUNTIME_S,
    slice_axes=list(SLICE_AXES) if RUN_SLICE_PROFILE else None,
)
(OUT / "run_config.json").write_text(json.dumps(run_config, indent=2, default=str))
if not KEEP_RAW:
    for p in (PORES_RAW, MASK_RAW): Path(p).unlink(missing_ok=True)
    (RAW_DIR / "raw_meta.json").unlink(missing_ok=True)
print("written to", OUT); [print("  ", p.relative_to(OUT)) for p in sorted(OUT.rglob("*")) if p.is_file() and "_raw" not in p.parts]

written to /home/alberto.vicente/phd/DataDriven_UT_AlbertoVicente/02_XCT_data/Fabricacion Nacho/05_Probetas_Nacho_2025/probetas/Na_01_1/onlypores files/pore_analysis
   figures/hist_phi_theta.png
   figures/hist_volume_diameter_sphericity_elongation.png
   figures/porosity_profile_slices.png
   figures/porosity_vs_z_slabs.png
   figures/projection_z.png
   figures/scatter_volume_sphericity_by_type.png
   pores.csv
   pores.parquet
   pores_raw.key.json
   pores_raw.parquet
   run_config.json
   slice_profile_x.csv
   slice_profile_x.key.json
   slice_profile_y.csv
   slice_profile_y.key.json
   slice_profile_z.csv
   slice_profile_z.key.json
   specimen_summary.json


[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

## 10. How to reuse

1. Edit only the **Parameters** cell: `PORES_VOLUME_PATH`, `MASK_VOLUME_PATH` (TIFF stacks, same shape, `(Z, Y, X)`), and **`VOXEL_SIZE_MM`** (must be set per specimen – the default 0.025 is just the placeholder agreed for Na_01_1). Use the `defect-atlas` kernel.
2. *Run All*. Re-runs are cheap: the TIFF→RAW conversion and the pipeline result are cached in `OUT_DIR` and invalidated automatically when the inputs, voxel size, size filters, merging flag or backend sources change (`USE_CACHE=False` forces a rerun).
3. Quick tests on a big scan: set `CROP=((z0,z1),(y0,y1),(x0,x1))` (note: VVF and border flags then refer to the cropped box).
4. Outputs in `OUT_DIR`: `pores.csv/.parquet` (one row per pore, no `coords`), `specimen_summary.json`, `slice_profile_*.csv`, `figures/*.png`, `run_config.json`. `_raw/` holds the normalised RAW copies (`KEEP_RAW=False` deletes them at the end, at the cost of reconverting next time).
5. Reading results later: `pd.read_parquet(OUT_DIR/"pores.parquet")`. Units: volumes µm³, lengths µm, `*_distance_to_surface` (no suffix) in **nm**, `*_vox` in voxels, `centroid-*`/`bbox-*` in voxels `(z, y, x)`.
6. Caveats: `Area3D_capped` / `cpx_factor_capped` flag pores whose surface/complexity value was capped; `shape_class` is dominated by the `cpx_factor` override (a voxelised sphere has cpx_factor ≈ 1.77, above the default 0.7001) – prefer `shape_class_no_cpx` for elongation-based grouping, or raise `CPX_THRESHOLD`. Phi/Theta and the covariance ratios are NaN or non-positive for 1-voxel / degenerate pores (no shape class for those); Sphericity can exceed 1 for tiny pores – set `MIN_SIZE_VOXELS` to drop them. The notebook never writes inside the Defect Atlas repo.